# Project 1 Full Workflow Code Notebook

Canonical one-file workflow for grading under the one-code-file assumption.

This notebook integrates: acquisition checks, cleaning/preprocessing, EDA, feature engineering, diagnostics, and ablation.


## 1) Imports and Configuration

In [ ]:
from __future__ import annotations

import json
import re
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim import corpora
from gensim.models import LdaModel
from scipy.stats import kruskal, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

SEED = 42
REQUIRED_RAW_COLUMNS = ["title", "body", "url", "score", "comms_num", "timestamp"]

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'Code Files' else Path.cwd().resolve()
if not (ROOT / 'Project Deliverables').exists():
    ROOT = Path('/Users/m2/Projects/STAT 5243/Project 1')

RAW_PATH = ROOT / 'Project Deliverables' / 'Datasets' / 'reddit_wsb.csv'
CLEANED_CANONICAL_PATH = ROOT / 'Project Deliverables' / 'Datasets' / 'reddit_wsb_cleaned.csv'
OUT_DIR = ROOT / 'Project Workspace' / 'Supporting Materials' / 'Generated Outputs' / 'one_code_notebook'
FIG_DIR = OUT_DIR / 'artifacts' / 'figures'
JSON_DIR = OUT_DIR / 'artifacts' / 'json'

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW_PATH exists:', RAW_PATH.exists())
print('CANONICAL CLEANED exists:', CLEANED_CANONICAL_PATH.exists())

## 2) Acquisition Integrity Checks

In [ ]:
raw = pd.read_csv(RAW_PATH)

missing_cols = [c for c in REQUIRED_RAW_COLUMNS if c not in raw.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

print('Raw shape:', raw.shape)
print('Columns:', len(raw.columns))
print('Missing body % (raw):', round(raw['body'].isna().mean() * 100, 4))

## 3) Cleaning and Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http[s]?://\S+", "", text)
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)
    text = re.sub(r"[*_]{1,3}", "", text)
    text = re.sub(r"#{1,6}\s*", "", text)
    text = (
        text.replace("&amp;", "&")
        .replace("&lt;", "<")
        .replace("&gt;", ">")
        .replace("&nbsp;", " ")
        .replace("&#x200B;", "")
    )
    return re.sub(r"\s+", " ", text).strip()


def infer_post_type(url: str) -> str:
    if not isinstance(url, str):
        return "other"
    u = url.lower()
    if "v.redd.it" in u or "youtube" in u or "youtu.be" in u:
        return "video"
    if "i.redd.it" in u or u.endswith((".png", ".jpg", ".jpeg", ".gif")):
        return "image"
    if "reddit.com/r/wallstreetbets/comments" in u:
        return "text"
    return "link"


def preprocess(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()
    for col in ["score", "comms_num"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["score", "comms_num"]).copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).copy()

    if "id" in df.columns:
        df = df.drop_duplicates(subset=["id"]).copy()
    else:
        df = df.drop_duplicates().copy()

    if "created" in df.columns:
        df = df.drop(columns=["created"])

    df["date"] = df["timestamp"].dt.date.astype(str)
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.day_name()

    df["title_clean"] = df["title"].fillna("").map(clean_text)
    df["body_clean"] = df["body"].fillna("").map(clean_text)
    df["title_nlp"] = df["title_clean"].str.lower()

    df["has_body"] = df["body"].fillna("").str.strip().ne("")
    df["title_length"] = df["title_clean"].str.len()

    df["post_type"] = df["url"].map(infer_post_type)
    freq = df["post_type"].value_counts(normalize=True)
    keep = set(freq[freq >= 0.05].index)
    df["post_type_lumped"] = df["post_type"].where(df["post_type"].isin(keep), "Other")

    df["score_log"] = np.log1p(df["score"].clip(lower=0))
    df["comms_num_log"] = np.log1p(df["comms_num"].clip(lower=0))

    for col in ["score_log", "comms_num_log", "title_length", "hour"]:
        mu = df[col].mean()
        sd = df[col].std(ddof=0)
        mn = df[col].min()
        mx = df[col].max()
        df[f"{col}_zscore"] = (df[col] - mu) / (sd if sd else 1.0)
        df[f"{col}_minmax"] = (df[col] - mn) / ((mx - mn) if (mx - mn) else 1.0)

    df["day_of_week_encoded"] = df["day_of_week"].astype("category").cat.codes
    df["type_image"] = df["post_type_lumped"].eq("image")
    df["type_text"] = df["post_type_lumped"].eq("text")
    return df

In [ ]:
cleaned = preprocess(raw)
print('Cleaned shape:', cleaned.shape)
print('Missing body % (cleaned):', round(cleaned['body'].isna().mean() * 100, 4))

cleaned_out = OUT_DIR / 'reddit_wsb_cleaned_full_workflow_code_notebook.csv'
cleaned.to_csv(cleaned_out, index=False)
print('Wrote:', cleaned_out)

## 4) EDA and Advanced Diagnostics

In [ ]:
def save_dist(df: pd.DataFrame, col: str, stem: str, bins: int = 50):
    plt.figure(figsize=(8, 4))
    plt.hist(df[col].dropna(), bins=bins)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig(FIG_DIR / stem, dpi=220)
    plt.close()


def run_eda(df: pd.DataFrame) -> dict:
    save_dist(df, 'score', '03_eda_fig_01_score_distribution.png')
    save_dist(df, 'comms_num', '03_eda_fig_02_comms_distribution.png')
    save_dist(df, 'score_log', '03_eda_fig_03_score_log_distribution.png')
    save_dist(df, 'comms_num_log', '03_eda_fig_04_comms_log_distribution.png')
    save_dist(df, 'title_length', '03_eda_fig_05_title_length_distribution.png')
    save_dist(df, 'hour', '03_eda_fig_06_hour_distribution.png', bins=24)

    plt.figure(figsize=(7, 5))
    plt.scatter(df['score_log'], df['comms_num_log'], s=4, alpha=0.25)
    plt.xlabel('score_log')
    plt.ylabel('comms_num_log')
    plt.title('score_log vs comms_num_log')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '03_eda_fig_10_score_comments_scatter.png', dpi=220)
    plt.close()

    dow = df.groupby('day_of_week', observed=False)['score_log'].mean().reindex([
        'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'
    ])
    plt.figure(figsize=(8, 4))
    dow.plot(kind='bar')
    plt.title('Mean score_log by day_of_week')
    plt.ylabel('Mean score_log')
    plt.tight_layout()
    plt.savefig(FIG_DIR / '03_eda_fig_14_score_dow_trend.png', dpi=220)
    plt.close()

    rho, pval = spearmanr(df['score_log'], df['comms_num_log'], nan_policy='omit')

    groups = [g['score_log'].dropna().values for _, g in df.groupby('post_type_lumped') if len(g) > 0]
    if len(groups) > 1:
        stat, p_kruskal = kruskal(*groups)
    else:
        stat, p_kruskal = float('nan'), float('nan')

    outlier_rate = float((df['score_log_zscore'].abs() >= 3).mean())

    return {
        'spearman_score_vs_comments': {'rho': float(rho), 'p_value': float(pval)},
        'kruskal_score_by_post_type': {'statistic': float(stat), 'p_value': float(p_kruskal)},
        'score_log_outlier_rate_abs_ge_3': outlier_rate,
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
    }

In [ ]:
eda_stats = run_eda(cleaned)
(JSON_DIR / '03_eda_advanced_stats.json').write_text(json.dumps(eda_stats, indent=2))
print(json.dumps(eda_stats, indent=2))

## 5) Feature Engineering, Diagnostics, and Ablation

In [ ]:
POS_WORDS = {
    'gain', 'moon', 'bull', 'buy', 'rocket', 'win', 'green', 'profit', 'up', 'long', 'calls', 'squeeze'
}
NEG_WORDS = {
    'loss', 'bear', 'sell', 'drop', 'down', 'red', 'bagholder', 'panic', 'short', 'puts', 'crash'
}


def tokenize(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    return re.findall(r'[A-Za-z]{2,}', text.lower())


def simple_sentiment(text: str) -> float:
    toks = tokenize(text)
    if not toks:
        return 0.0
    pos = sum(t in POS_WORDS for t in toks)
    neg = sum(t in NEG_WORDS for t in toks)
    return (pos - neg) / len(toks)


def build_topic_features(df: pd.DataFrame, text_col: str) -> tuple[pd.DataFrame, dict]:
    tokenized = df[text_col].fillna('').astype(str).map(tokenize)
    dictionary = corpora.Dictionary(tokenized)
    dictionary.filter_extremes(no_below=30, no_above=0.5, keep_n=2000)
    corpus = [dictionary.doc2bow(tokens) for tokens in tokenized]

    if len(dictionary) == 0:
        topic_df = pd.DataFrame(np.zeros((len(df), 5)), columns=[f'topic_{i}' for i in range(5)])
        top_terms = {f'topic_{i}': [] for i in range(5)}
        return topic_df, top_terms

    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=5, passes=3, random_state=SEED)
    topic_probs = []
    for bow in corpus:
        dist = lda.get_document_topics(bow, minimum_probability=0.0)
        dist_sorted = sorted(dist, key=lambda x: x[0])
        topic_probs.append([p for _, p in dist_sorted])

    topic_df = pd.DataFrame(topic_probs, columns=[f'topic_{i}' for i in range(5)])
    topic_df['dominant_topic'] = topic_df.values.argmax(axis=1)
    top_terms = {f'topic_{i}': [w for w, _ in lda.show_topic(i, topn=10)] for i in range(5)}
    return topic_df, top_terms

In [ ]:
work = cleaned.copy()
text_col = 'title_nlp' if 'title_nlp' in work.columns else 'title_clean'
work['sentiment_score'] = work[text_col].fillna('').astype(str).map(simple_sentiment)

topic_df, top_terms = build_topic_features(work, text_col)
work = pd.concat([work.reset_index(drop=True), topic_df.reset_index(drop=True)], axis=1)

threshold = float(work['score'].quantile(0.95))
work['viral_flag'] = (work['score'] >= threshold).astype(int)

sample = work.sample(min(20000, len(work)), random_state=SEED)
base_cols = ['score_log', 'comms_num_log', 'title_length', 'hour', 'score_log_zscore', 'comms_num_log_zscore']
topic_cols = [c for c in sample.columns if c.startswith('topic_')]
feature_sets = {
    'baseline': [c for c in base_cols if c in sample.columns],
    'baseline_plus_sentiment': [c for c in base_cols + ['sentiment_score'] if c in sample.columns],
    'baseline_plus_sentiment_topics': [c for c in base_cols + ['sentiment_score'] + topic_cols if c in sample.columns],
}

y = sample['viral_flag'].astype(int)
train_idx, test_idx = train_test_split(sample.index, test_size=0.25, random_state=SEED, stratify=y)
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

rows = []
for label, cols in feature_sets.items():
    X = sample[cols].fillna(0.0)
    model = LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear')
    model.fit(X.loc[train_idx], y_train)
    pred_prob = model.predict_proba(X.loc[test_idx])[:, 1]
    pred = (pred_prob >= 0.5).astype(int)
    rows.append({
        'model': label,
        'feature_count': int(len(cols)),
        'roc_auc': float(roc_auc_score(y_test, pred_prob)),
        'average_precision': float(average_precision_score(y_test, pred_prob)),
        'f1_at_0_5': float(f1_score(y_test, pred)),
    })

best_cols = feature_sets['baseline_plus_sentiment_topics']
X = sample[best_cols].fillna(0.0)
model = LogisticRegression(max_iter=1200, class_weight='balanced', solver='liblinear')
model.fit(X.loc[train_idx], y_train)
pred_prob = model.predict_proba(X.loc[test_idx])[:, 1]
pred = (pred_prob >= 0.5).astype(int)

fpr, tpr, _ = roc_curve(y_test, pred_prob)
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title('ROC curve')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_04_roc_curve.png', dpi=220)
plt.close()

prec, rec, _ = precision_recall_curve(y_test, pred_prob)
plt.figure(figsize=(6, 6))
plt.plot(rec, prec)
plt.title('Precision-Recall curve')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_05_pr_curve.png', dpi=220)
plt.close()

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.title('Confusion Matrix')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, int(cm[i, j]), ha='center', va='center')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_06_confusion_matrix.png', dpi=220)
plt.close()

plt.figure(figsize=(8, 4))
plt.hist(work['sentiment_score'], bins=50)
plt.title('Sentiment score distribution')
plt.tight_layout()
plt.savefig(FIG_DIR / '04_feature_fig_01_sentiment_distribution.png', dpi=220)
plt.close()

In [ ]:
ablation_df = pd.DataFrame(rows)
labels = ablation_df['model'].tolist()
x = np.arange(len(labels))
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, ablation_df['roc_auc'], width=width, label='ROC-AUC')
plt.bar(x, ablation_df['average_precision'], width=width, label='Avg Precision')
plt.bar(x + width, ablation_df['f1_at_0_5'], width=width, label='F1@0.5')
plt.xticks(x, labels, rotation=12, ha='right')
plt.ylim(0.0, 1.05)
plt.title('Feature ablation: baseline vs +sentiment vs +topics')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / '05_feature_ablation_comparison.png', dpi=220)
plt.close()

ablation_payload = {
    'generated_at': datetime.now().isoformat(),
    'seed': SEED,
    'virality_threshold': threshold,
    'train_size': int(len(train_idx)),
    'test_size': int(len(test_idx)),
    'models': rows,
    'figure': str(FIG_DIR / '05_feature_ablation_comparison.png'),
}
(JSON_DIR / '05_feature_ablation_table.json').write_text(json.dumps(ablation_payload, indent=2))

feature_summary = {
    'roc_auc': float(roc_auc_score(y_test, pred_prob)),
    'average_precision': float(average_precision_score(y_test, pred_prob)),
    'f1_at_0_5': float(f1_score(y_test, pred)),
    'virality_threshold': threshold,
    'topic_top_terms': top_terms,
}
(JSON_DIR / '04_feature_summary_metrics.json').write_text(json.dumps(feature_summary, indent=2))

print(ablation_df)
print('F1 lift (+topics vs baseline):', round(rows[2]['f1_at_0_5'] - rows[0]['f1_at_0_5'], 6))

## 6) Reproducibility and Export Manifest

In [ ]:
manifest = {
    'generated_at': datetime.now().isoformat(),
    'raw_path': str(RAW_PATH),
    'canonical_cleaned_reference': str(CLEANED_CANONICAL_PATH),
    'outputs': {
        'cleaned_csv': str(cleaned_out),
        'json_dir': str(JSON_DIR),
        'fig_dir': str(FIG_DIR),
    },
    'eda_json': str(JSON_DIR / '03_eda_advanced_stats.json'),
    'ablation_json': str(JSON_DIR / '05_feature_ablation_table.json'),
}
(JSON_DIR / 'full_workflow_code_notebook_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))